# RAG Competition — PDF 기반 질문 답변 시스템

## 개요
이 노트북은 PDF 문서를 기반으로 질문에 자동으로 답변하는 **RAG(Retrieval-Augmented Generation)** 파이프라인을 구현합니다.

### 파이프라인 흐름
```
PDF 로드 → 텍스트 청킹 → 벡터 임베딩 → FAISS 저장 → 질문 검색 → LLM 답변 생성
```

### 주요 기술 스택
- **PDF 파싱**: `pypdf`
- **텍스트 분할**: `langchain_text_splitters`
- **임베딩 모델**: OpenAI `text-embedding-3-small`
- **벡터 저장소**: FAISS (Facebook AI Similarity Search)
- **LLM**: OpenAI `gpt-4o-mini`
- **체인 구성**: LangChain LCEL (LangChain Expression Language)

## 0. 환경 설정 및 라이브러리 임포트

In [ ]:
import os
import json
import time
from pathlib import Path

# .env 파일에서 환경변수(API 키 등)를 자동으로 불러옴
from dotenv import load_dotenv
load_dotenv()

# ── PDF 텍스트 추출 ────────────────────────────────────────────────────────────
# pypdf: PDF 파일에서 페이지별 텍스트를 추출하는 라이브러리
from pypdf import PdfReader

# ── 텍스트 분할 ────────────────────────────────────────────────────────────────
# RecursiveCharacterTextSplitter: 문단 → 줄 → 문장 → 단어 순으로 재귀적으로 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── 임베딩 & 벡터 저장소 ──────────────────────────────────────────────────────
# OpenAIEmbeddings: 텍스트를 고차원 벡터로 변환 (의미 유사도 계산에 사용)
# ChatOpenAI: GPT 모델을 통해 최종 답변 생성
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# FAISS: Facebook AI Research에서 만든 고속 유사도 검색 벡터 저장소
from langchain_community.vectorstores import FAISS

# ── LangChain 핵심 컴포넌트 ────────────────────────────────────────────────────
# Document: 텍스트 + 메타데이터(출처, 페이지 번호 등)를 담는 기본 단위
from langchain_core.documents import Document

# ChatPromptTemplate: LLM에게 전달할 프롬프트 템플릿 생성
from langchain_core.prompts import ChatPromptTemplate

# StrOutputParser: LLM 출력을 순수 문자열로 파싱
from langchain_core.output_parsers import StrOutputParser

# RunnablePassthrough: 입력값을 그대로 다음 단계로 전달하는 패스스루 컴포넌트
from langchain_core.runnables import RunnablePassthrough

print("✅ 모든 라이브러리 임포트 완료")

## 1. 하이퍼파라미터 및 프롬프트 설정

> **튜닝 포인트**: `CHUNK_SIZE`, `CHUNK_OVERLAP`, `TOP_K` 값을 조절하면 검색 품질에 직접 영향을 줍니다.

In [ ]:
# ── 모델 설정 ──────────────────────────────────────────────────────────────────
# 임베딩 모델: 텍스트 → 벡터 변환 (1536차원, 비용 저렴)
EMBEDDING_MODEL = "text-embedding-3-small"

# LLM 모델: 검색된 문맥을 바탕으로 최종 답변 생성
# gpt-4o-mini: 속도/비용 효율이 높은 소형 모델
LLM_MODEL = "gpt-4o-mini"

# ── 청킹 설정 ──────────────────────────────────────────────────────────────────
# CHUNK_SIZE: 각 청크의 최대 문자 수
# - 너무 크면: 관련 없는 내용이 함께 검색됨 (노이즈 증가)
# - 너무 작면: 문맥이 잘려 답변 품질 저하
CHUNK_SIZE = 800

# CHUNK_OVERLAP: 인접 청크 간 겹치는 문자 수
# - 문장이 청크 경계에서 잘리는 것을 방지하여 문맥 연속성 유지
CHUNK_OVERLAP = 150

# ── 검색 설정 ──────────────────────────────────────────────────────────────────
# TOP_K: 질문당 검색할 청크 수
# - 많을수록 더 많은 문맥을 제공하지만 토큰 비용 증가
TOP_K = 5

# ── RAG 프롬프트 템플릿 ────────────────────────────────────────────────────────
# {context}: 검색된 관련 문서 청크들이 삽입됨
# {question}: 사용자 질문이 삽입됨
# 핵심 지시사항: 반드시 제공된 문맥만 사용, 없으면 "Not found" 반환
RAG_PROMPT_TEMPLATE = """You are an expert assistant answering questions based strictly on the provided context.

Context:
{context}

Question: {question}

Instructions:
- Answer concisely and accurately using only the context above.
- If the answer is not in the context, say "Not found in the document."
- Do not hallucinate or add external knowledge.

Answer:"""

print(f"✅ 설정 완료:")
print(f"   임베딩 모델  : {EMBEDDING_MODEL}")
print(f"   LLM 모델     : {LLM_MODEL}")
print(f"   청크 크기    : {CHUNK_SIZE}자")
print(f"   청크 겹침    : {CHUNK_OVERLAP}자")
print(f"   검색 개수(K) : {TOP_K}개")

## 2. STEP 1 — PDF 로드 및 텍스트 추출

각 페이지를 개별 `Document` 객체로 변환하며, 빈 페이지(이미지만 있는 페이지 등)는 자동으로 제외합니다.

In [ ]:
def load_pdf(pdf_path: str) -> list[Document]:
    """
    PDF 파일에서 텍스트를 추출하여 LangChain Document 리스트로 반환합니다.

    Parameters
    ----------
    pdf_path : str
        PDF 파일 경로

    Returns
    -------
    list[Document]
        각 페이지가 하나의 Document 객체인 리스트
        메타데이터에 'source'(파일경로)와 'page'(페이지 번호) 포함
    """
    reader = PdfReader(pdf_path)
    docs = []

    for page_num, page in enumerate(reader.pages, start=1):
        # 페이지에서 텍스트 추출 (텍스트가 없으면 빈 문자열 반환)
        text = page.extract_text() or ""
        text = text.strip()

        # 빈 페이지(이미지 전용, 스캔본 등) 건너뜀
        if text:
            docs.append(Document(
                page_content=text,
                metadata={
                    "source": pdf_path,   # 출처 파일 경로
                    "page": page_num      # 1-indexed 페이지 번호
                }
            ))

    print(f"[PDF 로드] 전체 {len(reader.pages)}페이지 → 텍스트 있는 페이지: {len(docs)}개")
    return docs


# ── 실행 ──────────────────────────────────────────────────────────────────────
# 기본 PDF 경로 설정 (필요시 변경)
PDF_PATH = "data/2025_SK하이닉스_지속가능경영보고서.pdf"

# PDF 파일 존재 여부 확인
if Path(PDF_PATH).exists():
    docs = load_pdf(PDF_PATH)
    # 첫 번째 페이지의 텍스트 일부를 미리보기로 출력
    if docs:
        print(f"\n📄 첫 번째 페이지 미리보기 (100자):")
        print(docs[0].page_content[:100], "...")
else:
    print(f"⚠️  PDF 파일을 찾을 수 없습니다: {PDF_PATH}")
    print("    PDF_PATH 변수를 올바른 경로로 수정해 주세요.")

## 3. STEP 2 — 텍스트 청킹 (문서 분할)

긴 페이지 텍스트를 LLM의 컨텍스트 윈도우에 맞는 작은 청크로 분할합니다.
`RecursiveCharacterTextSplitter`는 `\n\n → \n → . → 공백` 순서로 분리 지점을 탐색합니다.

In [ ]:
def split_documents(docs: list[Document]) -> list[Document]:
    """
    문서 리스트를 지정된 크기의 청크로 분할합니다.

    RecursiveCharacterTextSplitter 동작 원리:
    1. 먼저 '\n\n'(문단 구분)으로 분리 시도
    2. 청크가 여전히 크면 '\n'(줄 구분)으로 재분리
    3. 그래도 크면 '. '(문장 구분)으로 재분리
    4. 최후 수단으로 공백(' ')으로 분리

    Parameters
    ----------
    docs : list[Document]
        PDF에서 로드된 페이지 단위 Document 리스트

    Returns
    -------
    list[Document]
        분할된 청크 리스트 (원본 메타데이터 상속)
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,       # 청크 최대 길이 (문자 수 기준)
        chunk_overlap=CHUNK_OVERLAP, # 청크 간 겹침 (문맥 연속성 유지)
        separators=["\n\n", "\n", ". ", " ", ""],  # 분리 우선순위
    )

    chunks = splitter.split_documents(docs)

    print(f"[청킹 완료] {len(docs)}개 페이지 → {len(chunks)}개 청크 생성")
    print(f"           (청크 크기: {CHUNK_SIZE}자, 겹침: {CHUNK_OVERLAP}자)")

    # 청크 길이 분포 통계 출력
    lengths = [len(c.page_content) for c in chunks]
    print(f"           청크 길이 통계: 최소 {min(lengths)}자 / 평균 {sum(lengths)//len(lengths)}자 / 최대 {max(lengths)}자")

    return chunks


# ── 실행 ──────────────────────────────────────────────────────────────────────
chunks = split_documents(docs)

# 임의 청크 내용 확인
print(f"\n📌 샘플 청크 (인덱스 0):")
print(f"   페이지: {chunks[0].metadata.get('page')}페이지")
print(f"   내용  : {chunks[0].page_content[:150]}...")

## 4. STEP 3 — FAISS 벡터 저장소 구축

각 청크를 임베딩 벡터로 변환하고 FAISS 인덱스에 저장합니다.
이후 질문이 들어오면 코사인 유사도 기반으로 가장 관련성 높은 청크를 검색합니다.

> ⚠️ **비용 주의**: 이 단계에서 OpenAI Embeddings API가 호출됩니다 (청크 수만큼 요청 발생)

In [ ]:
def build_vectorstore(chunks: list[Document]) -> FAISS:
    """
    청크 리스트를 임베딩하여 FAISS 벡터 저장소를 생성합니다.

    처리 과정:
    1. OpenAI API로 각 청크 텍스트 → 1536차원 float 벡터 변환
    2. FAISS 인덱스(Flat L2)에 모든 벡터 저장
    3. 이후 쿼리 벡터와의 유사도 계산에 사용

    Parameters
    ----------
    chunks : list[Document]
        분할된 청크 Document 리스트

    Returns
    -------
    FAISS
        검색 가능한 FAISS 벡터 저장소 객체
    """
    print(f"[임베딩 시작] '{EMBEDDING_MODEL}' 모델로 {len(chunks)}개 청크 임베딩 중...")
    print("              (OpenAI API 호출 중, 잠시 기다려 주세요...)")

    # OpenAI 임베딩 모델 초기화
    # OPENAI_API_KEY 환경변수에서 자동으로 API 키를 읽어옴
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

    # 청크 리스트를 임베딩하여 FAISS 인덱스 생성
    # 내부적으로 배치 처리하여 API 요청 수를 최소화
    vectorstore = FAISS.from_documents(chunks, embeddings)

    print(f"[임베딩 완료] 벡터 저장소 준비됨 ({len(chunks)}개 벡터 저장)")
    return vectorstore


# ── OPENAI_API_KEY 설정 확인 ──────────────────────────────────────────────────
if not os.environ.get("OPENAI_API_KEY"):
    raise EnvironmentError(
        "❌ OPENAI_API_KEY 환경변수가 설정되지 않았습니다.\n"
        "   .env 파일에 OPENAI_API_KEY=sk-... 를 추가하거나\n"
        "   os.environ['OPENAI_API_KEY'] = 'sk-...' 로 직접 설정하세요."
    )

# ── 실행 ──────────────────────────────────────────────────────────────────────
vectorstore = build_vectorstore(chunks)

# 유사도 검색 테스트 (선택사항)
test_query = "지속가능경영 목표"
test_results = vectorstore.similarity_search(test_query, k=2)
print(f"\n🔍 테스트 검색: '{test_query}'")
for i, doc in enumerate(test_results, 1):
    print(f"   결과 {i} (페이지 {doc.metadata.get('page')}): {doc.page_content[:80]}...")

## 5. STEP 4 — RAG 체인 구성

LangChain LCEL(LangChain Expression Language)을 사용해 검색-생성 파이프라인을 연결합니다.

```
질문 → [retriever] → 관련 청크 목록 → [format_docs] → 문자열 문맥
     ↘                                                      ↗
       [RunnablePassthrough] → 질문 그대로 전달
                  ↓
         [prompt] → 채워진 프롬프트
                  ↓
             [ChatOpenAI] → LLM 응답
                  ↓
         [StrOutputParser] → 최종 문자열 답변
```

In [ ]:
def build_rag_chain(vectorstore: FAISS):
    """
    FAISS 벡터 저장소를 기반으로 RAG 체인을 구성합니다.

    구성 요소:
    - retriever  : 질문과 유사한 상위 K개 청크를 벡터 저장소에서 검색
    - format_docs: 검색된 Document 리스트를 하나의 문자열로 포맷팅
    - prompt     : 문맥 + 질문을 LLM에게 전달할 형식으로 조합
    - llm        : GPT 모델로 최종 답변 생성
    - parser     : LLM 출력 메시지를 순수 문자열로 변환

    Parameters
    ----------
    vectorstore : FAISS
        임베딩된 청크가 저장된 FAISS 벡터 저장소

    Returns
    -------
    tuple[chain, retriever]
        체인 (invoke로 실행)과 리트리버 (검색 결과 확인용)
    """
    # retriever: 벡터 저장소를 검색 인터페이스로 래핑
    # search_type 기본값은 'similarity' (코사인 유사도 기반 검색)
    retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

    # LLM 초기화: temperature=0으로 설정하여 결정론적(일관된) 답변 생성
    # temperature > 0 이면 같은 질문에도 매번 다른 답변이 나올 수 있음
    llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

    # 프롬프트 템플릿: {context}와 {question} 플레이스홀더를 포함
    prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)

    def format_docs(docs: list[Document]) -> str:
        """
        검색된 Document 리스트를 LLM이 읽기 좋은 형식의 문자열로 변환합니다.
        각 청크 앞에 페이지 번호를 표시하고 '---'로 구분합니다.
        """
        return "\n\n---\n\n".join(
            f"[Page {d.metadata.get('page', '?')}]\n{d.page_content}"
            for d in docs
        )

    # LCEL 체인 조립: | 연산자로 각 단계를 파이프라인처럼 연결
    # dict 형태로 context와 question을 병렬로 처리하여 prompt에 전달
    chain = (
        {
            # retriever로 관련 청크 검색 후 format_docs로 문자열 변환
            "context": retriever | format_docs,
            # 질문은 변환 없이 그대로 전달
            "question": RunnablePassthrough()
        }
        | prompt          # 문맥 + 질문 → 완성된 프롬프트
        | llm             # 프롬프트 → LLM 응답 (AIMessage 객체)
        | StrOutputParser()  # AIMessage → 순수 문자열
    )

    print("✅ RAG 체인 구성 완료")
    print(f"   검색 방식: 코사인 유사도, Top-{TOP_K} 청크 검색")
    print(f"   LLM: {LLM_MODEL} (temperature=0)")
    return chain, retriever


# ── 실행 ──────────────────────────────────────────────────────────────────────
chain, retriever = build_rag_chain(vectorstore)

## 6. STEP 5 — 질문 일괄 처리 및 답변 생성

질문 파일을 읽어 각 질문에 대해 RAG 체인을 실행하고 결과를 수집합니다.

In [ ]:
def answer_questions(
    questions: list[str],
    chain,
    retriever,
    delay: float = 0.3,
) -> list[dict]:
    """
    질문 리스트를 RAG 체인에 순차적으로 전달하여 답변을 생성합니다.

    각 질문에 대해:
    1. retriever로 관련 문서 청크 검색 (어떤 페이지에서 답변했는지 기록용)
    2. chain.invoke()로 최종 답변 생성
    3. 결과를 dict 형태로 수집

    Parameters
    ----------
    questions : list[str]
        답변할 질문 문자열 리스트
    chain : Runnable
        build_rag_chain()으로 생성한 RAG 체인
    retriever : VectorStoreRetriever
        검색 결과 확인용 리트리버
    delay : float
        질문 간 대기 시간(초) — OpenAI API 레이트 리밋 방지용

    Returns
    -------
    list[dict]
        각 항목: {id, question, answer, retrieved_pages}
    """
    results = []
    total = len(questions)

    for i, question in enumerate(questions, start=1):
        question = question.strip()

        # 빈 줄 건너뜀 (파일에 빈 줄이 있는 경우 대비)
        if not question:
            continue

        # 진행 상황 출력 (질문 앞 80자만 표시)
        print(f"[질문 {i}/{total}] {question[:80]}{'...' if len(question) > 80 else ''}")

        # ① 검색: 질문과 유사한 청크를 벡터 저장소에서 가져옴 (투명성 확보)
        retrieved_docs = retriever.invoke(question)
        # 참조된 페이지 번호 목록 추출 (결과 추적용)
        retrieved_pages = [d.metadata.get("page", "?") for d in retrieved_docs]

        # ② 생성: 검색된 문맥 + 질문을 LLM에 전달하여 답변 생성
        answer = chain.invoke(question)

        # 결과 저장
        results.append({
            "id": i,
            "question": question,
            "answer": answer.strip(),
            "retrieved_pages": retrieved_pages,  # 어떤 페이지를 참조했는지 기록
        })

        # API 레이트 리밋 방지: 연속 호출 시 과도한 요청을 막기 위해 짧게 대기
        if delay:
            time.sleep(delay)

    print(f"\n✅ 총 {len(results)}개 질문 처리 완료")
    return results

## 7. 질문 파일 로드 및 실행

질문 파일 경로를 설정하고 전체 파이프라인을 실행합니다.

In [ ]:
# ── 질문 파일 경로 설정 ────────────────────────────────────────────────────────
QUESTIONS_FILE = "questions.txt"  # 한 줄에 질문 하나씩 작성된 텍스트 파일
OUTPUT_FILE    = "answers.json"   # 결과를 저장할 JSON 파일 경로

# 질문 파일 존재 확인
if not Path(QUESTIONS_FILE).exists():
    raise FileNotFoundError(
        f"❌ 질문 파일을 찾을 수 없습니다: {QUESTIONS_FILE}\n"
        f"   QUESTIONS_FILE 변수를 올바른 경로로 수정해 주세요."
    )

# ── 질문 파일 읽기 ─────────────────────────────────────────────────────────────
with open(QUESTIONS_FILE, "r", encoding="utf-8") as f:
    # 각 줄을 질문으로 읽고, 빈 줄은 제외
    questions = [line.strip() for line in f if line.strip()]

print(f"[질문 로드] '{QUESTIONS_FILE}'에서 {len(questions)}개 질문 불러옴")
print("\n📋 질문 목록:")
for i, q in enumerate(questions, 1):
    print(f"   {i}. {q}")

In [ ]:
# ── RAG 파이프라인 실행 ────────────────────────────────────────────────────────
# 모든 질문에 대해 검색 + 생성을 순차 실행
results = answer_questions(questions, chain, retriever)

## 8. 결과 저장 및 확인

In [ ]:
# ── 출력 데이터 구조 구성 ─────────────────────────────────────────────────────
# metadata: 실행 조건을 기록하여 재현성 확보 및 결과 비교에 활용
output = {
    "metadata": {
        "pdf":              PDF_PATH,        # 사용한 PDF 파일 경로
        "questions_file":   QUESTIONS_FILE,  # 질문 파일 경로
        "llm_model":        LLM_MODEL,       # 답변 생성에 사용한 LLM
        "embedding_model":  EMBEDDING_MODEL, # 임베딩에 사용한 모델
        "chunk_size":       CHUNK_SIZE,      # 청킹 설정
        "chunk_overlap":    CHUNK_OVERLAP,   # 청크 겹침 설정
        "top_k":            TOP_K,           # 검색된 청크 수
        "total_questions":  len(results),    # 처리된 질문 수
    },
    "results": results,  # 각 질문별 답변 및 참조 페이지
}

# ── JSON 파일로 저장 ───────────────────────────────────────────────────────────
# ensure_ascii=False: 한글 등 유니코드 문자를 이스케이프 없이 저장
# indent=2: 사람이 읽기 쉬운 들여쓰기 형식
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"✅ 완료! {len(results)}개 답변이 '{OUTPUT_FILE}'에 저장되었습니다.")

# ── 결과 미리보기 ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 결과 미리보기 (처음 3개)")
print("="*60)
for r in results[:3]:
    print(f"\n[Q{r['id']}] {r['question']}")
    print(f"[A]  {r['answer'][:200]}{'...' if len(r['answer']) > 200 else ''}")
    print(f"[참조 페이지] {r['retrieved_pages']}")